# BioAI Evidence Validator — quickstart

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NingyuSUN/bioai-evidence-validator/blob/main/examples/quickstart.ipynb)

An LLM read a paper and extracted *"gene A is associated with phenotype A"*. Is that good enough for a
research summary? For a curated knowledge base? This notebook walks one claim from LLM extraction to
human-reviewed admission in about two minutes. All data is synthetic.

In [1]:
import subprocess, sys

try:
    from bioevidence_validator import build_record, draft_json_schema, validate_record
except ImportError:  # e.g. on Colab
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "bioai-evidence-validator>=0.5"])
    from bioevidence_validator import build_record, draft_json_schema, validate_record

## 1. Write the claim as a compact draft

A draft names each fact once. `build_record` derives identifiers, groups evidence into lines, and hashes
the local source file. It never invents facts: scope, extraction method and retrieval time are explicit.

In [2]:
from pathlib import Path

Path("paper.txt").write_text("Table 2: Synthetic gene A is associated with synthetic phenotype A.\n", encoding="utf-8")

draft = {
    "profile": "literature-claim",
    "uses": ["research_summary"],
    "statement": {
        "subject": {"id": "SYN:GENE_A", "label": "Synthetic gene A", "type": "gene"},
        "predicate": "associated_with",
        "object": {"id": "SYN:PHENOTYPE_A", "label": "Synthetic phenotype A", "type": "phenotype"},
        "scope": ["taxon:synthetic"],
    },
    "sources": [{
        "id": "paper", "title": "Synthetic paper", "type": "publication", "version": "v1",
        "retrieved_at": "2026-09-21T00:00:00Z", "file": "paper.txt",
    }],
    "evidence": [{
        "source": "paper", "locator": "Table 2", "type": "publication_result",
        "text": "Synthetic gene A is associated with synthetic phenotype A.",
        "method": "llm_extraction", "scope": ["taxon:synthetic"],
    }],
}


def check(draft):
    report = validate_record(build_record(draft), profile=draft["profile"])
    for decision in report["use_decisions"]:
        print(f"{decision['use']:>17}: {decision['admission_status']:<16} {', '.join(decision['reason_codes'])}")
    for finding in report["findings"]:
        print(f"{'':>19}{finding['rule_id']}: {finding['message']}")


check(draft)

 research_summary: review_required  BEV008
                   BEV008: Required evidence type 'publication_result' comes only from LLM extraction.


The LLM-only evidence is held for review (`BEV008`): the `literature-claim` profile does not let a
publication result rest on LLM extraction alone.

## 2. A curator confirms the evidence by hand

In [3]:
draft["evidence"][0]["method"] = "manual_curation"
check(draft)

 research_summary: admitted         


## 3. Ask for a stricter use

Knowledge-base admission in this profile also needs an explicit human acceptance.

In [4]:
draft["uses"] = ["research_summary", "knowledge_base"]
check(draft)

 research_summary: admitted         
   knowledge_base: rejected         BEV010
                   BEV010: This use requires explicit human acceptance.


In [5]:
draft["reviews"] = [{
    "reviewer": {"id": "orcid:0000-0000-0000-0000", "name": "Synthetic curator", "type": "human"},
    "decision": "accept", "uses": ["knowledge_base"],
    "rationale": "Checked Table 2 against the source.", "decided_at": "2026-09-22T00:00:00Z",
}]
check(draft)

 research_summary: admitted         
   knowledge_base: admitted         


## 4. Plug in your LLM

`draft_json_schema` turns a profile into a JSON Schema for structured output: allowed predicates,
entity types and uses become enums. Give it to your model, then **set `method` in your own pipeline code**:
a model should not declare how its own output was produced.

In [6]:
schema = draft_json_schema("literature-claim")
statement = schema["properties"]["statement"]["properties"]
print("predicates:", statement["predicate"]["enum"])
print("subject types:", statement["subject"]["properties"]["type"]["enum"])
print("uses:", schema["properties"]["uses"]["items"]["enum"])

predicates: ['associated_with', 'contributes_to']
subject types: ['gene', 'variant']
uses: ['knowledge_base', 'research_summary']


Next steps: [write your own profile](https://github.com/NingyuSUN/bioai-evidence-validator/blob/main/docs/PROFILES.md),
read the [draft format](https://github.com/NingyuSUN/bioai-evidence-validator/blob/main/docs/DRAFTS.md), or run the
check on every pull request with the [GitHub Action](https://github.com/NingyuSUN/bioai-evidence-validator#check-records-in-ci).